# PATH CONFIG

In [1]:
import os

print(os.getcwd())
if not os.getcwd().endswith("app"):
    os.chdir("../app")
    print(os.getcwd())

import pandas as pd
pd.set_option('display.max_rows', 500)
pd.set_option('display.max_columns', 500)

%load_ext autoreload
%autoreload 2
# %matplotlib inline

/home/turbotowerlnx/Documents/Master/TA/TA-Spanish-Esperanto-Translator/notebooks
/home/turbotowerlnx/Documents/Master/TA/TA-Spanish-Esperanto-Translator/app


In [2]:
from src.config import Configuration

CONFIG = Configuration(
    model_name="meta-llama/Llama-2-7b-hf",
    src_code = "spa_Latn",
    tgt_code = "epo_Latn"
)

# Dataset

In [ ]:
df_corpus_clean = pd.read_csv(CONFIG.corpus_path)
df_corpus_clean.rename(columns={
    CONFIG.src_name: CONFIG.src_code, 
    CONFIG.tgt_name: CONFIG.tgt_code
}, inplace=True)

# Shuffle the dataframe first
df_shuffled = df_corpus_clean.sample(frac=1, random_state=42).reset_index(drop=True)

n_total = len(df_shuffled)
n_test = int(n_total * CONFIG.test_split)
n_val = int(n_total * CONFIG.val_split)

df_test = df_shuffled[:n_test].reset_index(drop=True)
df_val = df_shuffled[n_test:n_test + n_val].reset_index(drop=True)
df_train = df_shuffled[n_test + n_val:].reset_index(drop=True)

print(f"Dataset sizes:")
print(f"  Train: {len(df_train)} ({len(df_train)/n_total*100:.1f}%)")
print(f"  Val:   {len(df_val)} ({len(df_val)/n_total*100:.1f}%)")
print(f"  Test:  {len(df_test)} ({len(df_test)/n_total*100:.1f}%)")
print(f"  Total: {n_total}")

Dataset sizes:
  Train: 4019270 (70.0%)
  Val:   861272 (15.0%)
  Test:  861272 (15.0%)
  Total: 5741814


### Reduce dataset size for experiments

In [ ]:
n_train = int(len(df_train)*CONFIG.data_fraction)
n_val = int(len(df_val)*CONFIG.data_fraction)
n_test = int(len(df_test)*CONFIG.data_fraction)

df_train = df_train[:n_train]
df_val = df_val[:n_val]
df_test = df_test[:n_test]

print(f"Using {n_train:_} samples for training, {n_val:_} for validation and {n_test:_} for testing.")

In [ ]:
import os
import dotenv
from huggingface_hub import login

dotenv.load_dotenv()
login(token=os.getenv("HUGGING_FACE_TOKEN"))

/home/turbotowerlnx/Documents/Master/TA/TA-Spanish-Esperanto-Translator/venv/lib/python3.12/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm
python-dotenv could not parse statement starting at line 2
python-dotenv could not parse statement starting at line 4
python-dotenv could not parse statement starting at line 5
python-dotenv could not parse statement starting at line 7
python-dotenv could not parse statement starting at line 8
python-dotenv could not parse statement starting at line 4
python-dotenv could not parse statement starting at line 5
python-dotenv could not parse statement starting at line 7
python-dotenv could not parse statement starting at line 8


In [ ]:
from transformers import AutoTokenizer
from src.data import TranslationDatasetLLM


tokenizer = AutoTokenizer.from_pretrained(
    CONFIG.model_name,
    use_auth_token=True, 
    padding=True, 
    pad_to_multiple_of=8, 
    # src_lang=CONFIG.src_code, 
    # tgt_lang=CONFIG.tgt_code, 
    truncation=True, 
    max_length=CONFIG.max_tok_length,
    padding_side='left',
)
tokenizer.pad_token = tokenizer.eos_token

dataloader_train = TranslationDatasetLLM(
    dataframe=df_train,
    tokenizer=tokenizer,
    src_col=CONFIG.src_code,
    tgt_col=CONFIG.tgt_code,
    task_prefix=CONFIG.task_prefix,
    max_length=CONFIG.max_tok_length,
)
dataloader_val = TranslationDatasetLLM(
    dataframe=df_val,
    tokenizer=tokenizer,
    src_col=CONFIG.src_code,
    tgt_col=CONFIG.tgt_code,
    task_prefix=CONFIG.task_prefix,
    max_length=CONFIG.max_tok_length,
)
dataloader_test = TranslationDatasetLLM(
    dataframe=df_test,
    tokenizer=tokenizer,
    src_col=CONFIG.src_code,
    tgt_col=CONFIG.tgt_code,
    task_prefix=CONFIG.task_prefix,
    max_length=CONFIG.max_tok_length,
)

/home/turbotowerlnx/Documents/Master/TA/TA-Spanish-Esperanto-Translator/venv/lib/python3.12/site-packages/transformers/models/auto/tokenization_auto.py:1025: FutureWarning: The `use_auth_token` argument is deprecated and will be removed in v5 of Transformers. Please use `token` instead.
  warnings.warn(


Using 4_019_270 samples for training, 861_272 for validation and 861_272 for testing.


The LLM dataset contains the 'prefix' for the task. This are just the instructions for the LLM

In [ ]:
CONFIG.task_prefix

'translate from Spanish to Esperanto: '

# Load transformer

In [ ]:
import torch
from transformers import BitsAndBytesConfig, AutoModelForCausalLM

quantization_config = BitsAndBytesConfig(
    load_in_4bit=True,
    bnb_4bit_quant_type='nf4',
    bnb_4bit_use_double_quant=True,
    bnb_4bit_compute_dtype=torch.bfloat16,
)

model = AutoModelForCausalLM.from_pretrained(
    CONFIG.model_name,
    use_auth_token=True,
    quantization_config=quantization_config,
    torch_dtype=torch.bfloat16,
)


/home/turbotowerlnx/Documents/Master/TA/TA-Spanish-Esperanto-Translator/venv/lib/python3.12/site-packages/transformers/models/auto/auto_factory.py:492: FutureWarning: The `use_auth_token` argument is deprecated and will be removed in v5 of Transformers. Please use `token` instead.
  warnings.warn(
`torch_dtype` is deprecated! Use `dtype` instead!
Loading checkpoint shards: 100%|██████████| 2/2 [00:04<00:00,  2.38s/it]



# Evaluation

In [ ]:
from evaluate import load

metric_bleu = load("sacrebleu")
metric_comet = load("comet")

/home/turbotowerlnx/Documents/Master/TA/TA-Spanish-Esperanto-Translator/venv/lib/python3.12/site-packages/torchmetrics/utilities/imports.py:23: UserWarning: pkg_resources is deprecated as an API. See https://setuptools.pypa.io/en/latest/pkg_resources.html. The pkg_resources package is slated for removal as early as 2025-11-30. Refrain from using this package or pin to Setuptools<81.
  from pkg_resources import DistributionNotFound, get_distribution
Fetching 5 files: 100%|██████████| 5/5 [00:00<00:00, 138884.24it/s]

Lightning automatically upgraded your loaded checkpoint from v1.8.3.post1 to v2.5.6. To apply the upgrade to your files permanently, run `python -m pytorch_lightning.utilities.upgrade_checkpoint ../../../../../.cache/huggingface/hub/models--Unbabel--wmt22-comet-da/snapshots/2760a223ac957f30acfb18c8aa649b01cf1d75f2/checkpoints/model.ckpt`
Lightning automatically upgraded your loaded checkpoint from v1.8.3.post1 to v2.5.6. To apply the upgrade to your files permanently, run `

In [ ]:
import numpy as np
def postprocess_text(preds, labels):
    preds = [pred.strip() for pred in preds]
    labels = [[label.strip()] for label in labels]

    return preds, labels

def compute_metrics(eval_preds):
    preds, labels, sources = eval_preds

    # Convert to lists if coming from a datasets.Column
    if not isinstance(labels, list):
        labels = list(labels)
        
    if isinstance(preds, tuple):
        preds = preds[0]
    
    # Decode predictions
    decoded_preds = tokenizer.batch_decode(preds, skip_special_tokens=True)

    # Replace negative ids in the labels as we can't decode them.
    labels = [
        [tokenizer.pad_token_id if j < 0 else j for j in label]
        for label in labels
    ]
    decoded_labels = tokenizer.batch_decode(labels, skip_special_tokens=True)
    
    # Decode sources
    decoded_sources = tokenizer.batch_decode(sources, skip_special_tokens=True)

    # Some simple post-processing
    decoded_preds, decoded_labels = postprocess_text(decoded_preds, decoded_labels)

    result_blue = metric_bleu.compute(
        predictions=decoded_preds, 
        references=decoded_labels
    )
    result_comet = metric_comet.compute(
        sources=decoded_sources,
        predictions=decoded_preds, 
        references=[label[0] for label in decoded_labels]  # COMET expects flat list, not nested
    )
    result = {
        "bleu": result_blue["score"],
        "comet": result_comet["mean_score"]
    }

    prediction_lens = [np.count_nonzero(pred != tokenizer.pad_token_id) for pred in preds]
    result["gen_len"] = np.mean(prediction_lens)
    result = {k: round(v, 4) for k, v in result.items()}
    return result

# Fine-tunning and training

In [ ]:
from peft import prepare_model_for_kbit_training
from peft import LoraConfig, get_peft_model
# from transformers import DataCollatorForSeq2Seq
from transformers import DataCollatorForLanguageModeling

model = prepare_model_for_kbit_training(
    model,
    use_gradient_checkpointing=False,
    gradient_checkpointing_kwargs={'use_reentrant':False}
)

config = LoraConfig(
    task_type="SEQ_2_SEQ_LM",
    r=16,
    lora_alpha=32,
    target_modules=["q_proj", "k_proj", "v_proj"],
    lora_dropout=0.05,
    bias="none",
)
lora_model = get_peft_model(model, config)
lora_model.print_trainable_parameters()

# data_collator = DataCollatorForSeq2Seq(
#     tokenizer=tokenizer, 
#     model=lora_model, 
#     pad_to_multiple_of=8
# )
data_collator = DataCollatorForLanguageModeling(tokenizer=tokenizer, mlm=False, pad_to_multiple_of=8)

trainable params: 12,582,912 || all params: 6,750,998,528 || trainable%: 0.1864


In [ ]:
# from transformers import Seq2SeqTrainingArguments, Seq2SeqTrainer
from transformers import TrainingArguments, Trainer

# args = Seq2SeqTrainingArguments(
#     logging_dir=CONFIG.logging_dir,
#     eval_strategy = "epoch",
#     learning_rate=1e-4,
#     per_device_train_batch_size=CONFIG.batch_size,
#     per_device_eval_batch_size=CONFIG.batch_size,
#     weight_decay=0.01,
#     save_total_limit=2,
#     num_train_epochs=2,
#     predict_with_generate=True,
# )

args = TrainingArguments(
    CONFIG.output_model_dir,
    eval_strategy = "epoch",
    save_strategy = "epoch",
    logging_strategy = "epoch",
    logging_steps = 50,
    logging_first_step = True,
    learning_rate=1e-4,
    per_device_train_batch_size=CONFIG.batch_size,
    per_device_eval_batch_size=CONFIG.batch_size,
    weight_decay=0.01,
    save_total_limit=2,
    num_train_epochs=3,
    warmup_steps=100,
    optim="adamw_bnb_8bit",
    prediction_loss_only=False,
    gradient_accumulation_steps = 1,
    bf16=True,
    bf16_full_eval=True,
    group_by_length=True,
    load_best_model_at_end=True,
    metric_for_best_model="epoch",
    greater_is_better=True,
    disable_tqdm=False,
)

from transformers import TrainerCallback

class PrintEpochProgressCallback(TrainerCallback):
    def on_step_end(self, args, state, control, **kwargs):
        try:
            epoch = state.epoch if state.epoch is not None else 0.0
            max_steps = state.max_steps if getattr(state, 'max_steps', None) else None
            remaining = (max_steps - state.global_step) if max_steps else 'unknown'
            print(f"[Training] epoch={epoch:.2f}  step={state.global_step}/{max_steps}  remaining={remaining}")
        except Exception as e:
            print(f"[Training] step={state.global_step} (error computing epoch: {e})")

trainer = Trainer(
    lora_model,
    args,
    train_dataset=dataloader_train,
    eval_dataset=dataloader_val,
    tokenizer=tokenizer,
    data_collator=data_collator,
    callbacks=[PrintEpochProgressCallback()],
    # compute_metrics=compute_metrics,
)


ValueError: --load_best_model_at_end requires the save and eval strategy to match, but found
- Evaluation strategy: IntervalStrategy.EPOCH
- Save strategy: SaveStrategy.STEPS

In [ ]:
trainer.train()

The tokenizer has new PAD/BOS/EOS tokens that differ from the model config and generation config. The model config and generation config were aligned accordingly, being updated with the tokenizer's values. Updated tokens: {'pad_token_id': 2}.
The model is already on multiple devices. Skipping the move to device specified in `args`.
/home/turbotowerlnx/Documents/Master/TA/TA-Spanish-Esperanto-Translator/venv/lib/python3.12/site-packages/transformers/tokenization_utils_base.py:4034: UserWarning: `as_target_tokenizer` is deprecated and will be removed in v5 of Transformers. You can tokenize your labels by using the argument `text_target` of the regular `__call__` method (either in the same call as your input texts if you use the same keyword arguments, or in a separate call.
  warnings.warn(


ZeroDivisionError: integer division or modulo by zero

# Inference

In [ ]:
from transformers import GenerationConfig

generation_config = GenerationConfig.from_pretrained(
    CONFIG.model_name,
)

print(generation_config)

GenerationConfig {
  "bos_token_id": 0,
  "decoder_start_token_id": 2,
  "eos_token_id": 2,
  "max_length": 200,
  "pad_token_id": 1
}



In [ ]:
from torch.utils.data import DataLoader

test_batch_size = 32
test_loader = DataLoader(dataloader_test, batch_size=test_batch_size, shuffle=False)

In [ ]:
output_sequences = []
all_labels = []
all_sources = []

for i, batch in enumerate(test_loader):
    # Store source input_ids for later decoding
    all_sources.extend(batch['input_ids'].cpu())
    
    # Generate translations
    with torch.no_grad():    
        output_batch = model.generate(
            generation_config=generation_config, 
            input_ids=batch['input_ids'].cuda(), 
            attention_mask=batch['attention_mask'].cuda(), 
            forced_bos_token_id=tokenizer.convert_tokens_to_ids(CONFIG.tgt_code), 
            max_length=CONFIG.max_tok_length, 
            num_beams=1, 
            do_sample=False,
        )
    output_sequences.extend(output_batch.cpu())
    all_labels.extend(batch['labels'].cpu())
    
    if (i + 1) % 10 == 0:
        print(f"Processed {i + 1}/{len(test_loader)} batches")
    if i >= CONFIG.max_batches:
        break

/home/turbotowerlnx/Documents/Master/TA/TA-Spanish-Esperanto-Translator/venv/lib/python3.12/site-packages/transformers/tokenization_utils_base.py:4034: UserWarning: `as_target_tokenizer` is deprecated and will be removed in v5 of Transformers. You can tokenize your labels by using the argument `text_target` of the regular `__call__` method (either in the same call as your input texts if you use the same keyword arguments, or in a separate call.
  warnings.warn(


Processed 10/26915 batches
Processed 20/26915 batches
Processed 20/26915 batches
Processed 30/26915 batches
Processed 30/26915 batches
Processed 40/26915 batches
Processed 40/26915 batches
Processed 50/26915 batches
Processed 50/26915 batches
Processed 60/26915 batches
Processed 60/26915 batches
Processed 70/26915 batches
Processed 70/26915 batches
Processed 80/26915 batches
Processed 80/26915 batches
Processed 90/26915 batches
Processed 90/26915 batches
Processed 100/26915 batches
Processed 100/26915 batches
Processed 110/26915 batches
Processed 110/26915 batches
Processed 120/26915 batches
Processed 120/26915 batches
Processed 130/26915 batches
Processed 130/26915 batches
Processed 140/26915 batches
Processed 140/26915 batches
Processed 150/26915 batches
Processed 150/26915 batches
Processed 160/26915 batches
Processed 160/26915 batches
Processed 170/26915 batches
Processed 170/26915 batches
Processed 180/26915 batches
Processed 180/26915 batches
Processed 190/26915 batches
Processed

In [ ]:
# Compute metrics
result = compute_metrics((output_sequences, all_labels, all_sources))
print(f'BLEU score: {result["bleu"]}')
print(f'COMET score: {result["comet"]}')

💡 Tip: For seamless cloud uploads and versioning, try installing [litmodels](https://pypi.org/project/litmodels/) to enable LitModelCheckpoint, which syncs automatically with the Lightning model registry.
GPU available: True (cuda), used: True
TPU available: False, using: 0 TPU cores
/home/turbotowerlnx/Documents/Master/TA/TA-Spanish-Esperanto-Translator/venv/lib/python3.12/site-packages/torch/__init__.py:1551: UserWarning: Please use the new API settings to control TF32 behavior, such as torch.backends.cudnn.conv.fp32_precision = 'tf32' or torch.backends.cuda.matmul.fp32_precision = 'ieee'. Old settings, e.g, torch.backends.cuda.matmul.allow_tf32 = True, torch.backends.cudnn.allow_tf32 = True, allowTF32CuDNN() and allowTF32CuBLAS() will be deprecated after Pytorch 2.9. Please see https://pytorch.org/docs/main/notes/cuda.html#tensorfloat-32-tf32-on-ampere-and-later-devices (Triggered internally at /pytorch/aten/src/ATen/Context.cpp:80.)
  return _C._get_float32_matmul_precision()
You a

BLEU score: 19.4891
COMET score: 0.7691


# Examples

In [ ]:
from maikol_utils.print_utils import print_separator
from src.utils import decode_list, clean_sequence

decoded_sources = decode_list(all_sources)
decoded_outputs = decode_list(output_sequences)

for source, output in zip(decoded_sources, decoded_outputs):
    print(clean_sequence(source))
    print(clean_sequence(output))
    print_separator()

 ¡instaura nuevos hábitos!
 Eklaboru novajn kutimojn!
________________________________________________________________
 abran sus ojos, abran sus oídos, pues ha llegado
 Malfermu viajn okulojn, malfermu viajn orelojn,
________________________________________________________________
 iremos mal.
 Ni maltrafos la vojon.
________________________________________________________________
 muchos paquetes de linux tienen versiones raras.
 Multaj Linux-paketoj havas rarajn versiojn.
________________________________________________________________
 porque se dio cuenta de que la iglesia está en ruinas por la
 ĉar li rimarkis ke la preĝejo estas ruiniĝ
________________________________________________________________
 el asteroide 5996, lleva el nombre julioangel, en su
 la asteroido 5996, nomata Julio Angel, en sia
________________________________________________________________
 antes de tomar el cuadrado, perry decidió irse y volver
 Antaŭ ol preni la kvadrato, Perry decidis foriri kaj re
____